# 23 - Human-in-the-Loop

Human-in-the-loop (HITL) means an agent pauses before or during an important action so a person can approve, edit, reject, or provide input.

Current LangChain/LangGraph documentation supports HITL middleware and interrupts for tool-call review, with checkpointing so execution can pause and resume safely. citeturn114600search1turn114600search2

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# %pip install -qU langchain langgraph langchain-openai

## 1. Where HITL belongs

Use human review for high-impact actions:
- database writes
- financial transactions
- sending messages
- deleting data
- production changes
- sensitive external actions

## 2. Simple approval state

Before executing a sensitive tool, create a review request.

In [ ]:
approval_request = {
    'tool': 'send_email',
    'args': {
        'recipient': 'customer@example.com',
        'subject': 'Account update',
    },
    'risk': 'external side effect',
    'status': 'pending_review',
}

print(approval_request)

## 3. LangGraph interrupt concept

LangGraph's `interrupt()` pauses execution and surfaces a value to the caller. Resumption uses `Command`. A checkpointer is required because the state must be persisted across the pause. citeturn114600search1

In [ ]:
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class State(TypedDict):
    action: str
    approved: bool

def approval_node(state: State):
    decision = interrupt({
        'question': 'Approve this action?',
        'action': state['action'],
    })
    return {'approved': bool(decision)}

builder = StateGraph(State)
builder.add_node('approval', approval_node)
builder.add_edge(START, 'approval')
builder.add_edge('approval', END)

graph = builder.compile(checkpointer=InMemorySaver())

config = {'configurable': {'thread_id': 'approval-1'}}

# First call pauses at the interrupt:
# result = graph.invoke(
#     {'action': 'send_email', 'approved': False},
#     config
# )
# print(result)
#
# Resume later:
# result = graph.invoke(
#     Command(resume=True),
#     config
# )
# print(result)

## 4. Current LangChain HITL middleware

Current LangChain also exposes `HumanInTheLoopMiddleware`, where tools can be configured with approval policies such as approve/edit/reject/respond. It requires checkpointing for pause/resume workflows. citeturn114600search2

In [ ]:
from langchain.agents import create_agent

# Example shape:
#
# from langchain.agents.middleware import HumanInTheLoopMiddleware
# from langgraph.checkpoint.memory import InMemorySaver
#
# agent = create_agent(
#     model=CHAT_MODEL,
#     tools=[send_email, read_data],
#     middleware=[
#         HumanInTheLoopMiddleware(
#             interrupt_on={
#                 'send_email': True,
#                 'read_data': False,
#             }
#         )
#     ],
#     checkpointer=InMemorySaver(),
# )

## 5. Human decisions

Useful decisions:
- approve
- edit
- reject
- respond with extra information

Do not treat every action equally. Read-only operations usually need less friction than irreversible actions.

## 6. Evaluation and practice

Evaluate:
- correct intervention rate
- false-positive approvals
- time to resume
- action safety
- what happens when the human rejects

Practice: add HITL to a fake `delete_file` tool.

## Evaluation

HITL is part of safety, not decoration.

Improve with:
- risk-based policies
- clear review messages
- editable arguments
- durable checkpointing
- audit logs
- explicit timeout/expiry policies.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/